In [ ]:
import pandas as pd

# Đường dẫn đến file TSV
file_path = r'F:\Data Analyst\Voc Translation And Sentiment Pipeline\raw_data\amazon_reviews_multilingual_US_v1_00.tsv'

# Tên cột marketplace trong dataset Amazon
marketplace_column = 'marketplace'

# Số dòng đọc trong mỗi chunk (có thể điều chỉnh tùy thuộc vào RAM)
chunk_size = 100000

# Series dùng để lưu trữ tổng số review của từng marketplace
total_marketplace_counts = pd.Series(dtype=int)

# Đọc file theo từng chunk (thêm on_bad_lines='skip' để bỏ qua các dòng lỗi format)
for chunk in pd.read_csv(file_path, sep='\t', chunksize=chunk_size, on_bad_lines='skip', low_memory=False, encoding='utf-8'):
    if marketplace_column in chunk.columns:
        # Đếm số lượng review theo marketplace trong chunk hiện tại
        chunk_counts = chunk[marketplace_column].value_counts()
        
        # Cộng dồn kết quả vào Series tổng
        total_marketplace_counts = total_marketplace_counts.add(chunk_counts, fill_value=0)

# Sắp xếp kết quả từ cao xuống thấp
total_marketplace_counts = total_marketplace_counts.sort_values(ascending=False)

print(total_marketplace_counts)


In [10]:

import pandas as pd
from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException

# Lấy một mẫu dữ liệu (sample) nhỏ để test vì file 3GB chạy qua từng dòng sẽ mất rất nhiều thời gian
file_path = r'F:\Data Analyst\Voc Translation And Sentiment Pipeline\raw_data\amazon_reviews_multilingual_US_v1_00.tsv'

# Đọc 10.000 dòng đầu tiên để phân tích
df_sample = pd.read_csv(file_path, sep='\t', nrows=10000, on_bad_lines='skip', low_memory=False, encoding='utf-8')

# Xử lý các ký tự bị mã hóa dưới dạng HTML (ví dụ: &quot;, &eacute;)
import html
df_sample['review_headline'] = df_sample['review_headline'].apply(lambda x: html.unescape(str(x)) if pd.notna(x) else x)
df_sample['review_body'] = df_sample['review_body'].apply(lambda x: html.unescape(str(x)) if pd.notna(x) else x)
df_sample['review_body'] = df_sample['review_body'].str.replace('\x1a', '?', regex=False)
df_sample['review_headline'] = df_sample['review_headline'].str.replace('\x1a', '?', regex=False)

def detect_language(text):
    if pd.isna(text) or str(text).strip() == '':
        return 'unknown'
    try:
        # Hàm detect sẽ trả về mã ngôn ngữ (ví dụ: 'en', 'fr', 'de')
        return detect(str(text))
    except LangDetectException:
        return 'unknown'

# Gộp headline và body lại để có nhiều text hơn, nhận diện sẽ chính xác hơn
df_sample['full_review'] = df_sample['review_headline'].fillna('') + " " + df_sample['review_body'].fillna('')

print("Đang nhận diện ngôn ngữ trên 10.000 dòng đầu, vui lòng đợi...")
df_sample['detected_lang'] = df_sample['full_review'].apply(detect_language)

# Thống kê số lượng từng ngôn ngữ
lang_counts = df_sample['detected_lang'].value_counts()

print("\nThống kê các ngôn ngữ được sử dụng:")
print(lang_counts)


Đang nhận diện ngôn ngữ trên 10.000 dòng đầu, vui lòng đợi...

Thống kê các ngôn ngữ được sử dụng:
detected_lang
en    9978
es      13
fr       4
de       2
pt       1
nl       1
it       1
Name: count, dtype: int64


In [ ]:
# Lọc ra các dòng có ngôn ngữ được nhận diện là tiếng Pháp (fr)
df_fr = df_sample[df_sample['detected_lang'] == 'fr']

# In ra tiêu đề và nội dung của các đánh giá tiếng Pháp
print(f"Tìm thấy {len(df_fr)} đánh giá bằng tiếng Pháp. Dưới đây là nội dung chi tiết:\n")
for index, row in df_fr.iterrows():
    print(f"--- Đánh giá {index} ---")
    print(f"Tiêu đề: {row['review_headline']}")
    print(f"Nội dung: {row['review_body']}\n")

# Hiển thị dưới dạng bảng DataFrame
df_fr[['marketplace', 'star_rating', 'review_headline', 'review_body']]


In [11]:
# Lọc ra các dòng có ngôn ngữ được nhận diện là tiếng Pháp (fr)
df_fr = df_sample[df_sample['detected_lang'] == 'es']

# In ra tiêu đề và nội dung của các đánh giá tiếng Pháp
print(f"Tìm thấy {len(df_fr)} đánh giá bằng tiếng es. Dưới đây là nội dung chi tiết:\n")
for index, row in df_fr.iterrows():
    print(f"--- Đánh giá {index} ---")
    print(f"Tiêu đề: {row['review_headline']}")
    print(f"Nội dung: {row['review_body']}\n")

# Hiển thị dưới dạng bảng DataFrame
df_fr[['marketplace', 'star_rating', 'review_headline', 'review_body']]

Tìm thấy 13 đánh giá bằng tiếng es. Dưới đây là nội dung chi tiết:

--- Đánh giá 556 ---
Tiêu đề: !La receta para exito!
Nội dung: 1. Hecha una copa de pasion con dos cucharadas de fuego.   2. Entonces adiciona un toque de petalos de rosas,(rojo preferible).  3. Pensando en el unico amor de tu vida, mezcla los ingredientes en su corazon usando to alma como cuchara por unas horas.   4. Quedate feliz en la memoria que has creado por dos o tres semanas.  5. Recomienda el libro a otra persona!

--- Đánh giá 3969 ---
Tiêu đề: Insuperable
Nội dung: Desde que el video de "Gimme the Power " se transmitio en MTV busque por dias en Internet hasta que encontre el MP3 en un FTP mexicano, donde además estaba el resto del CD, despues de tomar esta canción me dio curiosidad y baje otra ... "Mas vale cholo que mal acompañado" me encanto el estilo, luego baje "Cerdo" la cual me parecio increible y decidi bajar todas..y TODAS SON BUENAS. Obviamente, cuando el disco llego a El Salvador lo compre de inmed

,marketplace,star_rating,review_headline,review_body
556,US,5,!La receta para exito!,1. Hecha una copa de pasion con dos cucharadas...
3969,US,5,Insuperable,"Desde que el video de ""Gimme the Power "" se tr..."
4124,US,5,"Sentimiento, sencillez, arte ..., este albun e...",Despu?s de buscar por muchas partes del mundo ...
6158,US,5,Dirty Mind fue el principio,"Aunque es su tercer ?lbum, Dirty Mind es el pr..."
7340,US,5,yo creo que aerosmith es lo mejor en el mundo,bueno por empezar me parece buenisimo porque t...
7781,US,5,100% Sucio,(EN ESPA?OL) Los Sonic Youth no fueron ni ser?...
7952,US,5,es un buen compact,de esto si que debemos estar orgullosos de que...
7954,US,5,"Donald va Solo, y le va muy bien!",La revista Rolling Stone afirmó que si el firm...
7959,US,5,"Paul, eres lo máximo",Paul siempre fue el Beatle incomprendido; prin...
7961,US,3,Los Beatles en Blanco,"Por ejemplo, he aquí un álbum sobreestimado. E..."
